# Generation of curated data from "VA_Tract_EQI_PLACES_merged.csv"
### 1. The following columns are removed with given reasons:
   * Duplicated columns
   * `Education_area.1` - cannot find information on it
   * `Mean_drought` - all 0
   * `RUCA1_EQI` ~ `RUCA5_EQI` - principal components from existing columns, large proportion of missing values.

   This produces a intermediate dataset "VA_Tract_EQI_PLACES_merged_cleaned0.csv".

### 2. Data curation starting from "VA_Tract_EQI_PLACES_merged_cleaned0.csv".

In [3]:
# Import necessary modules
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

In [12]:
# Read the dataset (.csv) into a Pandas dataframe
df = pd.read_csv("/content/VA_Tract_EQI_PLACES_merged_cleaned0.csv")
print("Dataset loaded successfully. First 5 rows:")
df.head()

Dataset loaded successfully. First 5 rows:


,tract,StateAbbr,State,StateCode,County,CountyCode,CountyFIPS,RUCA_pri_cat,pop,sqmile,...,PLACES_HIGHCHOL,PLACES_KIDNEY,PLACES_LPA,PLACES_MAMMOUSE,PLACES_MHLTH,PLACES_OBESITY,PLACES_PHLTH,PLACES_SLEEP,PLACES_STROKE,PLACES_TEETHLOST
0,51001090100,VA,Virginia,51,Accomack,1,51001,5,2941,7.154468,...,41.2,3.5,25.8,77.0,12.5,33.4,13.9,33.5,4.2,12.7
1,51001090200,VA,Virginia,51,Accomack,1,51001,4,6156,72.512230,...,38.4,3.6,29.7,78.7,14.6,38.5,15.1,39.1,4.5,17.2
2,51001090300,VA,Virginia,51,Accomack,1,51001,5,2335,49.587524,...,40.3,4.0,34.6,77.2,16.9,40.5,18.4,40.0,5.2,22.4
3,51001090400,VA,Virginia,51,Accomack,1,51001,5,6234,74.795433,...,37.4,4.2,37.4,79.2,17.5,43.5,18.5,42.7,5.6,27.2
4,51001090500,VA,Virginia,51,Accomack,1,51001,5,2849,25.459696,...,37.4,3.8,33.7,77.6,16.3,40.5,17.0,40.7,4.9,21.8


In [13]:
df.shape

(1845, 65)

**This `df` can be used for visualization**

In [7]:
df.columns

Index(['tract', 'StateAbbr', 'State', 'StateCode', 'County', 'CountyCode',
       'CountyFIPS', 'RUCA_pri_cat', 'pop', 'sqmile', 'pop_dens', 'facilities',
       'facilities_area', 'facilities_area_log', 'pest', 'pest_log', 'bs_25yo',
       'time2work', 'crime_index', 'crime_index_log', 'education',
       'education_area', 'education_area_log', 'food_ratio', 'nindex_open',
       'nata_sum', 'nata_sum_log', 'ozone', 'pm25', 'poverty', 'poverty_log',
       'rent_income_pct', 'selfservice', 'selfservice_log', 'unemployment',
       'unemployment_log', 'OverallEQI', 'PLACES_ACCESS2', 'PLACES_ARTHRITIS',
       'PLACES_BINGE', 'PLACES_BPHIGH', 'PLACES_BPMED', 'PLACES_CANCER',
       'PLACES_CASTHMA', 'PLACES_CERVICAL', 'PLACES_CHD', 'PLACES_CHECKUP',
       'PLACES_CHOLSCREEN', 'PLACES_COLON_SCREEN', 'PLACES_COPD',
       'PLACES_COREM', 'PLACES_COREW', 'PLACES_CSMOKING', 'PLACES_DENTAL',
       'PLACES_DIABETES', 'PLACES_HIGHCHOL', 'PLACES_KIDNEY', 'PLACES_LPA',
       'PLACES_MAMMOUSE

### 3. Select predictor and outcome columns by not selecting columns that are derived from other columns.

In [15]:
# Select the columns to keep
cols_to_keep = ['RUCA_pri_cat', 'pop_dens', 'facilities_area', 'pest', 'bs_25yo', 'time2work', 'crime_index',
                       'education_area', 'food_ratio', 'nindex_open', 'nata_sum', 'ozone', 'pm25',
                       'poverty', 'rent_income_pct', 'selfservice', 'unemployment', 'PLACES_ACCESS2',
                       'PLACES_ARTHRITIS', 'PLACES_BINGE', 'PLACES_BPHIGH', 'PLACES_BPMED','PLACES_CANCER',
                       'PLACES_CASTHMA', 'PLACES_CERVICAL', 'PLACES_CHD', 'PLACES_CHECKUP',
                       'PLACES_CHOLSCREEN', 'PLACES_COLON_SCREEN', 'PLACES_COPD', 'PLACES_COREM',
                       'PLACES_COREW', 'PLACES_CSMOKING', 'PLACES_DENTAL', 'PLACES_DIABETES',
                       'PLACES_HIGHCHOL', 'PLACES_KIDNEY', 'PLACES_LPA', 'PLACES_MAMMOUSE',
                       'PLACES_MHLTH', 'PLACES_OBESITY', 'PLACES_PHLTH', 'PLACES_SLEEP', 'PLACES_STROKE',
                       'PLACES_TEETHLOST']

data = df[cols_to_keep]

### 4. Data splitting
Split the entire dataset (X, y) into training/validation and test sets at a 0.8:0.2 ratio, stratified by the `RUCA_pri_cat` column, to ensure a proportional representation of its categories in both sets.

**Choose your own outcome variable from here**

In [17]:
# Select the numerical predictor variables
predictor_variables = ['pop_dens', 'facilities_area', 'pest', 'bs_25yo', 'time2work', 'crime_index',
                       'education_area', 'food_ratio', 'nindex_open', 'nata_sum', 'ozone', 'pm25',
                       'poverty', 'rent_income_pct', 'selfservice', 'unemployment', 'PLACES_ACCESS2',
                       'PLACES_ARTHRITIS', 'PLACES_BINGE', 'PLACES_BPHIGH', 'PLACES_BPMED',
                       'PLACES_CASTHMA', 'PLACES_CERVICAL', 'PLACES_CHD', 'PLACES_CHECKUP',
                       'PLACES_CHOLSCREEN', 'PLACES_COLON_SCREEN', 'PLACES_COPD', 'PLACES_COREM',
                       'PLACES_COREW', 'PLACES_CSMOKING', 'PLACES_DENTAL', 'PLACES_DIABETES',
                       'PLACES_HIGHCHOL', 'PLACES_KIDNEY', 'PLACES_LPA', 'PLACES_MAMMOUSE',
                       'PLACES_MHLTH', 'PLACES_OBESITY', 'PLACES_PHLTH', 'PLACES_SLEEP', 'PLACES_STROKE',
                       'PLACES_TEETHLOST']

X_numerical = data[predictor_variables]

# One-hot encode the 'RUCA_pri_cat' categorical variable/column
X_RUCA_onehot = pd.get_dummies(data['RUCA_pri_cat'], prefix='RUCA', drop_first=True)

# Concatenate the numerical predictors with the one-hot encoded categorical columns
X = pd.concat([X_numerical, X_RUCA_onehot], axis=1)

# Define the target variable y
y = data['PLACES_CANCER']

print("Shape of feature set X:", X.shape)
print("Shape of target variable y:", y.shape)
print("First 5 rows of X:")
print(X.head())
print("First 5 rows of y:")
print(y.head())

Shape of feature set X: (1845, 47)
Shape of target variable y: (1845,)
First 5 rows of X:
     pop_dens  facilities_area        pest  bs_25yo  time2work  crime_index  \
0  411.071808         0.000000   11.176707     27.0       17.6           56   
1   84.896019         0.082745  344.039331     24.3       19.5          244   
2   47.088455         0.040333  125.706110      6.7       19.2           81   
3   83.347336         0.000000  330.874284      8.2       23.8           75   
4  111.902359         0.039278  240.178312     17.6       17.1           60   

   education_area  food_ratio  nindex_open  nata_sum  ...  PLACES_MHLTH  \
0    4.193184e-01    0.340909    92.692253  0.988358  ...          12.5   
1    9.653544e-02    0.454545    68.456841  1.094157  ...          14.6   
2    2.016636e-02    0.285714    84.836143  1.117513  ...          16.9   
3    9.358860e-02    0.153846    65.198860  1.026764  ...          17.5   
4    3.520000e-19    0.250000    70.192398  1.095621  ...   

In [18]:
# Split the data into training/validation and test sets, stratified by 'RUCA_pri_cat'
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=df['RUCA_pri_cat'], random_state=42)

print("Shape of X_train_val:", X_train_val.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train_val:", y_train_val.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train_val: (1476, 47)
Shape of X_test: (369, 47)
Shape of y_train_val: (1476,)
Shape of y_test: (369,)


### 5. Check for missing data and perform imputation

In [21]:
# Check missing data distribution
null_perc = data.isnull().sum() * 100 / len(X_train_val)
null_perc = null_perc.sort_values(ascending = False)
null_perc.to_frame(name='NaN %')

,NaN %
PLACES_COREW,0.271003
PLACES_COREM,0.203252
PLACES_TEETHLOST,0.203252
PLACES_MAMMOUSE,0.067751
RUCA_pri_cat,0.000000
time2work,0.000000
crime_index,0.000000
education_area,0.000000
food_ratio,0.000000
pop_dens,0.000000


In [30]:
# We only need to do imputation for predictor datasets but have to do it separately to prevent leakage
from sklearn.impute import SimpleImputer

# Instantiate SimpleImputer to handle potential NaN values in scaled data
imputer = SimpleImputer(strategy='mean')

# Impute missing values in X_scaled and convert to DataFrame
X_train_val_impute = pd.DataFrame(imputer.fit_transform(X_train_val), columns=X_train_val.columns)
X_test_impute = pd.DataFrame(imputer.fit_transform(X_test), columns=X_test.columns)

# Make sure the shapes do not change
print("Shape of X_train_val:", X_train_val.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of X_train_val_impute:", X_train_val_impute.shape)
print("Shape of X_test_impute:", X_test_impute.shape)
print("Shape of y_train_val:", y_train_val.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train_val: (1476, 47)
Shape of X_test: (369, 47)
Shape of X_train_val_impute: (1476, 47)
Shape of X_test_impute: (369, 47)
Shape of y_train_val: (1476,)
Shape of y_test: (369,)


### 6. Normalize/Scale Predictor Variables


In [33]:
from sklearn.preprocessing import StandardScaler

# Instantiate StandardScaler
scaler = StandardScaler()

# Fit the scaler to predictor dataframes and transform them
X_train_val_impute_scaled = pd.DataFrame(scaler.fit_transform(X_train_val_impute), columns=X_train_val_impute.columns)
X_test_impute_scaled = pd.DataFrame(scaler.fit_transform(X_test_impute), columns=X_test_impute.columns)

print("Shape of X_train_val_impute_scaled:", X_train_val_impute_scaled.shape)
print("Shape of X_test_impute_scaled:", X_test_impute_scaled.shape)
print("First 5 rows of X_train_val_impute_scaled:")
X_train_val_impute_scaled.head()

Shape of X_train_val_impute_scaled: (1476, 47)
Shape of X_test_impute_scaled: (369, 47)
First 5 rows of X_train_val_impute_scaled:


,pop_dens,facilities_area,pest,bs_25yo,time2work,crime_index,education_area,food_ratio,nindex_open,nata_sum,...,PLACES_MHLTH,PLACES_OBESITY,PLACES_PHLTH,PLACES_SLEEP,PLACES_STROKE,PLACES_TEETHLOST,RUCA_2,RUCA_3,RUCA_4,RUCA_5
0,-0.453488,-0.254274,-0.157164,-0.169272,-0.565345,0.660175,-0.424164,0.958302,0.673695,-1.240402,...,-1.202405,-0.997424,-0.008080,-1.812583,1.159995,-0.307046,-0.402746,-0.251795,-0.229743,-0.194871
1,0.328713,-0.254274,-0.431243,-1.086418,-0.368498,2.256125,-0.424164,1.259251,-1.542595,-0.393648,...,0.499961,2.109546,0.643910,2.475657,1.464351,1.240785,-0.402746,-0.251795,-0.229743,-0.194871
2,-0.400958,-0.254274,0.288238,-0.015638,1.403128,-0.582823,-0.339244,0.108566,0.017980,0.601528,...,-0.056581,-0.043657,-0.306909,-0.251104,-0.513961,-0.260611,-0.402746,-0.251795,-0.229743,-0.194871
3,-0.069751,-0.254274,-0.325027,1.441553,-0.480982,-1.104576,-0.277997,-1.105343,-0.025486,0.814848,...,-1.333356,-0.650600,-1.067563,-0.810440,-0.970495,-1.127397,-0.402746,-0.251795,-0.229743,-0.194871
4,0.390394,-0.254274,-0.431165,0.161273,-0.649708,-0.352638,-0.286827,-0.621676,-1.141726,-0.742458,...,-0.514911,-0.347128,-0.551405,-0.157882,-0.513961,-0.740439,-0.402746,-0.251795,-0.229743,-0.194871


### 7. Define Stratified K-Fold (K=5) Cross-Validation
Define a 5-fold cross-validation strategy on the training/validation set, stratifying by the 'RUCA_pri_cat' column. This will generate indices for 5 distinct train-validation splits while also ensuring each fold has a similar distribution of 'RUCA_pri_cat' categories.

**Alternatively, we can choose to do this without stratification**

In [ ]:
# Convert 'RUCA_pri_cat' to integer labels for stratification
# Ensure we only get the 'RUCA_pri_cat' values corresponding to the X_train_val indices
ruca_stratify_labels_train_val = df.loc[X_train_val_impute_scaled.index, 'RUCA_pri_cat'].astype('category').cat.codes

# Instantiate StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_cv_indices = skf.split(X_train_val_impute_scaled, ruca_stratify_labels_train_val)

print("StratifiedKFold instance created with 5 splits, shuffled, and random_state=42.")
print("RUCA stratification labels prepared for X_train_val.")

StratifiedKFold instance created with 5 splits, shuffled, and random_state=42.
RUCA stratification labels prepared for X_train_val.
